In [1]:
!pip install nltk

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [2]:
!pip install torch

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [3]:
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import pandas as pd
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from pathlib import Path
from MetaFeature import getfeature


In [4]:
df = pd.read_csv('./dataset/data.csv')
target_cols = [f'target{i}' for i in range(7)]
# 标签编码成数字并记录映射关系
all_label_mapping = []
for col in target_cols:
    df[col], unique = pd.factorize(df[col])
    all_label_mapping.append(unique)

In [5]:

# 预处理数据
def preprocess(text):
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    return lemmatized_tokens
 
# 获取模型输入， 使用Word2Vec 将Target和DatasetName转换为编码
def get_input(df, use_word2vec=True):
    for col_name in ['DatasetName', 'Target']:
        # 将所有数据集中的文本预处理并转换为列表的列表
        all_tokens = [preprocess(text) for text in df[col_name]]
        all_tokens = [[text] for text in df[col_name]]

        # 训练Word2Vec模型
        if use_word2vec:
            model = Word2Vec(all_tokens, vector_size=100, window=5, min_count=1, workers=4)
            model.train(all_tokens, total_examples=len(all_tokens), epochs=10)

            all_vectors = []
            for row in df[col_name].values:
                # 获取单词的向量表示
                word_vector = model.wv[row]
                all_vectors.append(word_vector)
            all_vectors = pd.DataFrame(all_vectors)
            all_vectors.columns = [f'{col_name}{i}' for i in all_vectors.columns]

            df = pd.concat([df, all_vectors], axis=1)
        
    # df['DatasetName'] = df['DatasetName'].astype('category').cat.codes
    # df['Target'] = df['Target'].astype('category').cat.codes

    return df.drop(['DatasetName', 'Target'], axis=1)

df = get_input(df, use_word2vec=True)

In [6]:
# 按type分组五折lightgbm，用来验证模型效果
from sklearn.model_selection import StratifiedKFold
n_fold = 5
spliter = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=0)
metric_df = pd.DataFrame(data=0, index=target_cols, columns=['accuracy', 'precision', 'recall', 'f1'])
models = []
for i, (train_index, test_index) in enumerate(spliter.split(df, df['type'])):
    train = df.iloc[train_index]
    test = df.iloc[test_index]
    X_train, y_train = train.drop(target_cols, axis=1), train[target_cols]
    X_test, y_test = test.drop(target_cols, axis=1), test[target_cols]
    classfier = VotingClassifier(estimators=[
        ('lr', LogisticRegression()),
        ('rf', RandomForestClassifier()),
        ('lgb', lgb.LGBMClassifier(n_estimators=1000, num_leaves=31, max_depth=7, learning_rate=0.05, n_jobs=-1))
    ], n_jobs=-1)
    model = MultiOutputClassifier(classfier)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred = pd.DataFrame(y_pred, columns=target_cols)
    for col in target_cols:
        accuracy_score = classification_report(y_test[col], y_pred[col], output_dict=True)['accuracy']
        precision, recall, f1, _ = classification_report(y_test[col], y_pred[col], output_dict=True)['weighted avg'].values()
        metric_df.loc[col] += [accuracy_score, precision, recall, f1]
metric_df /= n_fold

C:\Users\28556\AppData\Local\Temp\ipykernel_39692\1924016107.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  metric_df.loc[col] += [accuracy_score, precision, recall, f1]
C:\Users\28556\AppData\Local\Temp\ipykernel_39692\1924016107.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.905952380952381' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  metric_df.loc[col] += [accuracy_score, precision, recall, f1]
C:\Users\28556\AppData\Local\Temp\ipykernel_39692\1924016107.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.9' has dtype incompatible with int64, please explicitly cast to a compatible dtyp

In [7]:
print("最优指标：")
metric_df.max()

最优指标：


accuracy     0.930000
precision    0.894915
recall       0.930000
f1           0.904977
dtype: float64

In [11]:
print("平均指标：")
metric_df.mean()

平均指标：


accuracy     0.733810
precision    0.693201
recall       0.733810
f1           0.693767
dtype: float64

In [9]:
# # 预测新数据，对新数据进行一定的处理，然后使用Word2Vec编码，lightgbm预测
# def predict_new_data(dataset_name, target_name, dataset_type):
#     new_orign_df = pd.read_csv(f'./dataset/new_data/{dataset_name}.csv')
#     new_tensor = getfeature(new_orign_df)
#     new_numpy = new_tensor.mean(0).numpy()
    
#     new_df = pd.DataFrame(columns=[str(i) for i in range(7)])
#     new_df.loc[0] = new_numpy
#     new_df['DatasetName'] = dataset_name
#     new_df['Target'] = target_name
#     new_df['type'] = dataset_type
#     new_df['size0'] = new_df.shape[0]
#     new_df['size1'] = new_df.shape[1]
#     new_df = get_input(new_df)

#     X_train, y_train = df.drop(target_cols, axis=1), df[target_cols]
#     X_test = new_df
#     classfier = VotingClassifier(estimators=[
#         ('lr', LogisticRegression()),
#         ('rf', RandomForestClassifier()),
#         ('lgb', lgb.LGBMClassifier(n_estimators=1000, num_leaves=31, max_depth=7, learning_rate=0.05, n_jobs=-1))
#     ], n_jobs=-1)
#     model = MultiOutputClassifier(classfier)
#     model.fit(X_train, y_train)
#     y_pred = model.predict(X_test)

#     str_preds = []
#     for i, pred in enumerate(y_pred[0]):
#         label_mapping = all_label_mapping[i]
#         pred = label_mapping[pred]
#         str_preds.append(pred)
#     print(f'{dataset_name}的预测结果为{str_preds}')
#     # return str_preds

In [10]:
predict_new_data('airlines', 'Delay', 0)

NameError: name 'predict_new_data' is not defined

In [ ]:
predict_new_data('cpu_act', 'usr', 1)